# 06b · Monitor operativo near-real-time con bfastmonitor en R

Aplica `bfast::bfastmonitor()` (Verbesselt et al., 2012) sobre las series mensuales combinadas Landsat 8 + Sentinel-2 de cada estación, con una ventana histórica 2013-2019 y un periodo de monitoreo 2020-2025.

El modelo ajusta una descomposición armónica (`harmon` de orden 3) sobre la ventana histórica, considerada estable tras la rehabilitación hidráulica de canales (1996-1998), y evalúa el periodo de monitoreo en busca de breakpoints al nivel de significancia p < 0,05.

**Entrada:** `outputs/tables/ndvi_combinado_2013_2025.csv`  
**Salida:** `outputs/tables/bfastmonitor_estaciones.csv` (estación · breakpoint · magnitud · estado)

Esta salida alimenta directamente la lógica del semáforo del Nivel 2 del Digital Twin, integrada con `src/python/alertas_manglar.py`.

In [ ]:
library(bfast)
library(dplyr)

ROOT <- "/home/rstudio/work/proyecto-cgsm"

serie <- read.csv(file.path(ROOT, "outputs/tables/ndvi_combinado_2013_2025.csv"))
serie$fecha <- as.Date(paste0(serie$fecha, "-01"))

cat("Registros cargados:", nrow(serie), "\n")
cat("Estaciones:", length(unique(serie$estacion)), "\n")
cat("Rango temporal:", as.character(min(serie$fecha)), "a", as.character(max(serie$fecha)), "\n")

## Ajuste de bfastmonitor por estación

Para cada estación se construye un objeto `ts` mensual a partir de 2013, se separa la ventana histórica 2013-2019 del periodo de monitoreo 2020-2025 y se aplica el algoritmo. La detección reporta la fecha del primer breakpoint significativo, su magnitud y el estado actual del monitor.

In [ ]:
estaciones <- unique(serie$estacion)

resultados <- lapply(estaciones, function(est) {
  sub <- subset(serie, estacion == est)
  sub <- sub[order(sub$fecha), ]

  # Construir ts mensual desde 2013-01 con frecuencia 12
  start_year  <- as.integer(format(min(sub$fecha), "%Y"))
  start_month <- as.integer(format(min(sub$fecha), "%m"))
  ts_est <- ts(sub$ndvi, start = c(start_year, start_month), frequency = 12)

  # bfastmonitor con history 2013-2019, monitor desde 2020-01
  mon <- tryCatch(
    bfastmonitor(ts_est,
                 start   = c(2020, 1),
                 formula = response ~ harmon,
                 order   = 3,
                 level   = 0.05),
    error = function(e) NULL
  )

  if (is.null(mon)) {
    return(data.frame(estacion = est, breakpoint = NA, magnitud = NA, estado = "sin_modelo"))
  }

  bp_time <- if (!is.na(mon$breakpoint)) as.character(round(mon$breakpoint, 3)) else NA
  magn    <- if (!is.null(mon$magnitude)) mon$magnitude else NA
  estado  <- if (is.na(mon$breakpoint)) "estable" else "breakpoint_detectado"

  data.frame(estacion   = est,
             breakpoint = bp_time,
             magnitud   = magn,
             estado     = estado)
})

tabla <- do.call(rbind, resultados)
print(tabla)

## Guardar tabla de salida

In [ ]:
out_path <- file.path(ROOT, "outputs/tables/bfastmonitor_estaciones.csv")
write.csv(tabla, out_path, row.names = FALSE)
cat("Salida guardada en:", out_path, "\n")
cat("Filas:", nrow(tabla), "\n")